# 11 — TF/TPU Hazırlık Oturumu (ÜCRETSİZ CPU çalışma zamanı)

**Runtime: CPU seçin** — kredi harcamaz. Tek seferlik üç iş yapar:
1. Teacher ağırlıklarını framework-bağımsız `.npz`'ye aktarır → `Drive/AFETSONAR/tf_export/`
2. GATE torch referansını üretir (test_v3'ün ilk 200 satırı)
3. xBD split'lerini TFRecord shard'larına çevirir → `Drive/AFETSONAR/tfrecords/`

Toplam süre: ~3-5 saat (çoğu Drive I/O — bekleyen işlem, başında durmak gerekmez).
Bittiğinde `10_tf_tpu_training.ipynb` TPU oturumunda çalıştırılabilir.

Not: Bu oturumda torch (export + referans) ve tensorflow (TFRecord yazımı)
birlikte kullanılır ama transformers yalnızca torch tarafında (5.x) gerekir —
TFRecord dönüştürücüsü transformers import etmez, çakışma yoktur.

In [ ]:
# Drive bağla
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/AFETSONAR")
assert DRIVE.exists(), "AFETSONAR klasörü bulunamadı"

In [ ]:
# tf-port branch'ini klonla ve paketi kur (~3 dk)
import os
if not os.path.exists("/content/calamitas-ai"):
    !git clone -b tf-port https://github.com/Runc-Dev/calamitas-ai.git /content/calamitas-ai
else:
    !cd /content/calamitas-ai && git pull
%cd /content/calamitas-ai
!pip install -q -e .
!pip install -q "transformers==5.7.0"  # checkpoint'ların doğrulandığı sürüm

In [ ]:
# 1) Ağırlık export'u (~5 dk: 201 MB kopya + npz yazımı)
CKPT = "/content/calamitas-ai/checkpoints/teacher_v4_best_ema.pth"
!mkdir -p /content/calamitas-ai/checkpoints
if not os.path.exists(CKPT):
    !cp "{DRIVE}/checkpoints/teacher/teacher_v4_best_ema.pth" "{CKPT}"

!python scripts/export_weights_npz.py --checkpoint "{CKPT}"

!mkdir -p "{DRIVE}/tf_export"
!cp checkpoints/export/*.npz checkpoints/export/manifest.json "{DRIVE}/tf_export/"
!cp tests_tf/golden/loss_inputs.npz tests_tf/golden/loss_values.json "{DRIVE}/tf_export/" 2>/dev/null || true
!ls -lh "{DRIVE}/tf_export/"

In [ ]:
# 2) GATE torch referansı — test_v3'ün İLK 200 satırı (deterministik)
# CPU'da yavaştır (~1-2 saat) ama ücretsizdir ve beklemeden çalışır.
import pandas as pd
test_csv = DRIVE / "data/splits/test_v3.csv"
df = pd.read_csv(test_csv).head(200)
df.to_csv("/content/test200.csv", index=False)
df.to_csv(f"{DRIVE}/tf_export/test200.csv", index=False)
print(f"test200.csv: {len(df)} satır")

!python scripts/evaluate.py \
    --model "{CKPT}" \
    --test-csv /content/test200.csv \
    --batch-size 2 --device cpu \
    --output /content/gate_ref.json
!cp /content/gate_ref.json "{DRIVE}/tf_export/"
import json; print(json.load(open("/content/gate_ref.json"))["metrics"]["miou_no_bg"])

In [ ]:
# 3a) GATE + VAL shard'ları (küçük — önce bunlar, TPU oturumu erken başlayabilsin)
!mkdir -p "{DRIVE}/tfrecords"

!python scripts_tf/convert_to_tfrecords.py \
    --csv /content/test200.csv --split gate \
    --out-dir "{DRIVE}/tfrecords"

!python scripts_tf/convert_to_tfrecords.py \
    --csv "{DRIVE}/data/splits/val_v3.csv" --split val \
    --out-dir "{DRIVE}/tfrecords"

In [ ]:
# 3b) TRAIN shard'ları + çevrimdışı Copy-Paste (EN UZUN adım: ~2-4 saat Drive I/O)
!python scripts_tf/convert_to_tfrecords.py \
    --csv "{DRIVE}/data/splits/train_v3.csv" --split train \
    --copy-paste \
    --out-dir "{DRIVE}/tfrecords"

!ls "{DRIVE}/tfrecords" | head -20
!du -sh "{DRIVE}/tfrecords"
print("HAZIRLIK TAMAM — şimdi 10_tf_tpu_training.ipynb TPU oturumunda çalıştırılabilir")